In [2]:
# import the built-in library for reading/parsing JSON files
import json

# import pandas, used later to turn the parsed rows into a table (DataFrame)
import pandas as pd

# import Path, a convenient way to build and work with file/folder paths
from pathlib import Path

# define the folder where the hMOF JSON files live (D: drive, accessed via WSL's /mnt/d/)
json_folder = Path("/mnt/d/MOf/hMOF-10 1039 C2EE23201D-CarbonDioxide-mofdb-version_dc8a0295db")

# find every file in that folder ending in .json, and sort them alphabetically/numerically
json_files = sorted(json_folder.glob("*.json"))

# print how many JSON files were found, as a quick sanity check before processing them
print(f"Found {len(json_files)} JSON files")

Found 32768 JSON files


In [7]:
# reuse the standard CO2 pressures reported in this dataset
CO2_PRESSURES = [0.01, 0.05, 0.1, 0.5, 2.5]

# function to parse one JSON file into a flat dict of structural + CO2 features
def parse_one_mof(json_path):
    # open the JSON file for reading
    with open(json_path) as f:
        # load the JSON content into a Python dictionary
        data = json.load(f)

    # build the base row with structural properties pulled straight from the JSON
    row = {
        # use the filename (without extension) as a unique identifier
        "filename": Path(json_path).stem,
        # largest cavity diameter
        "lcd": data.get("lcd"),
        # pore limiting diameter
        "pld": data.get("pld"),
        # fraction of accessible void volume
        "void_fraction": data.get("void_fraction"),
        # gravimetric surface area (m^2/g)
        "surface_area_m2g": data.get("surface_area_m2g"),
        # SMILES string describing the linker, used later for RDKit features
        "mofid": data.get("mofid"),
    }

    # pre-create one CO2 uptake column per standard pressure, default to missing
    for p in CO2_PRESSURES:
        row[f"CO2_uptake_{p}bar_molkg"] = None

    # loop through every isotherm entry recorded for this MOF
    for entry in data.get("isotherms", []):
        # get the list of adsorbate gases for this isotherm entry
        adsorbates = entry.get("adsorbates", [])
        # skip this entry if there's no adsorbate listed, or it isn't CO2
        if not adsorbates or adsorbates[0].get("name") != "CarbonDioxide":
            continue
        # skip this entry if the units aren't mol/kg (we don't want kJ/mol heat data here)
        if entry.get("adsorptionUnits") != "mol/kg":
            continue
        # loop through each individual pressure/uptake data point in this isotherm
        for point in entry.get("isotherm_data", []):
            # pull out the pressure and the corresponding total adsorption value
            p, uptake = point.get("pressure"), point.get("total_adsorption")
            # check this pressure against each of our standard target pressures
            for target_p in CO2_PRESSURES:
                # if the pressure is close enough to a standard value, record the uptake
                if p is not None and abs(p - target_p) < 1e-3:
                    row[f"CO2_uptake_{target_p}bar_molkg"] = uptake
    # return the completed row for this one MOF
    return row

# test on just the first 20 files, not all 32,768 yet
sample_files = json_files[:20]
# run the parser on each sample file, building a list of row dictionaries
rows = [parse_one_mof(fp) for fp in sample_files]
# convert the list of rows into a pandas DataFrame (table)
df = pd.DataFrame(rows)
# display the resulting table
df

,filename,lcd,pld,void_fraction,surface_area_m2g,mofid,CO2_uptake_0.01bar_molkg,CO2_uptake_0.05bar_molkg,CO2_uptake_0.1bar_molkg,CO2_uptake_0.5bar_molkg,CO2_uptake_2.5bar_molkg
0,hMOF-0,11.75,10.75,0.795539,3676.3,[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn...,0.022406,0.073552,0.225851,0.885221,4.33307
1,hMOF-1,4.75,3.25,0.388986,580.7,[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn...,0.173407,0.925813,1.806490,4.779010,7.34618
2,hMOF-10,4.75,3.75,0.307505,377.6,[O-]C(=O)c1cc(F)c(c(c1F)F)C(=O)[O-].[Zn][O]([Z...,0.651164,2.129830,2.817220,3.909470,4.93236
3,hMOF-100,9.25,8.25,0.731919,3399.2,COc1cc(cc(c1C(=O)[O-])OC)C(=O)[O-].COc1cc(ccc1...,0.031321,0.143123,0.332034,1.756380,6.91190
4,hMOF-1000,3.75,2.75,0.323803,85.2,CCc1cc(C(=O)[O-])c(c(c1C(=O)[O-])CC)CC.[O-]C(=...,0.144024,0.698367,1.402230,2.694200,4.15694
5,hMOF-10000,3.75,2.75,0.194771,17.7,[O-]C(=O)c1cc(Br)c2c(c1)ccc(c2)C(=O)[O-].[O-]C...,0.239830,0.600202,1.482840,2.374780,2.61176
6,hMOF-10001,6.25,5.25,0.651894,2440.8,[O-]C(=O)C(=O)[O-].[O-]C(=O)c1ccc2c(c1)ccc(c2C...,0.062156,0.390535,0.552799,4.145720,9.93608
7,hMOF-10002,3.25,2.75,0.135421,0.0,[O-]C(=O)C(=O)[O-].[O-]C(=O)c1ccc2c(c1)ccc(c2C...,0.020168,0.077557,0.100751,0.373208,1.25613
8,hMOF-10003,12.25,10.75,0.818676,3893.8,[O-]C(=O)c1cc2ccc(c(c2c(c1C)C)C)C(=O)[O-].[O-]...,0.020223,0.116735,0.321263,1.261200,5.07845
9,hMOF-10004,6.25,3.75,0.523720,1089.3,[O-]C(=O)c1cc2ccc(c(c2c(c1C)C)C)C(=O)[O-].[O-]...,0.198655,1.007990,1.778510,4.220500,6.78177


In [8]:
# build the output file path where the test CSV will be saved
output_path = Path("../data/raw/hmof_test_20.csv")

# write the DataFrame to a CSV file, without including the pandas row index as a column
df.to_csv(output_path, index=False)

# print a confirmation showing how many rows were saved and where
print(f"Saved {len(df)} rows to {output_path}")

Saved 20 rows to ../data/raw/hmof_test_20.csv


In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv('/home/susan/MOF_project/data/raw/hmof_test_20.csv')
df.head()

,filename,lcd,pld,void_fraction,surface_area_m2g,mofid,CO2_uptake_0.01bar_molkg,CO2_uptake_0.05bar_molkg,CO2_uptake_0.1bar_molkg,CO2_uptake_0.5bar_molkg,CO2_uptake_2.5bar_molkg
0,hMOF-0,11.75,10.75,0.795539,3676.3,[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn...,0.022406,0.073552,0.225851,0.885221,4.33307
1,hMOF-1,4.75,3.25,0.388986,580.7,[O-]C(=O)c1ccc(cc1)C(=O)[O-].[Zn][O]([Zn])([Zn...,0.173407,0.925813,1.806490,4.779010,7.34618
2,hMOF-10,4.75,3.75,0.307505,377.6,[O-]C(=O)c1cc(F)c(c(c1F)F)C(=O)[O-].[Zn][O]([Z...,0.651164,2.129830,2.817220,3.909470,4.93236
3,hMOF-100,9.25,8.25,0.731919,3399.2,COc1cc(cc(c1C(=O)[O-])OC)C(=O)[O-].COc1cc(ccc1...,0.031321,0.143123,0.332034,1.756380,6.91190
4,hMOF-1000,3.75,2.75,0.323803,85.2,CCc1cc(C(=O)[O-])c(c(c1C(=O)[O-])CC)CC.[O-]C(=...,0.144024,0.698367,1.402230,2.694200,4.15694


In [6]:
print(df.shape)
print(df.columns)
print(df.info())

(20, 11)
Index(['filename', 'lcd', 'pld', 'void_fraction', 'surface_area_m2g', 'mofid',
       'CO2_uptake_0.01bar_molkg', 'CO2_uptake_0.05bar_molkg',
       'CO2_uptake_0.1bar_molkg', 'CO2_uptake_0.5bar_molkg',
       'CO2_uptake_2.5bar_molkg'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   filename                  20 non-null     str    
 1   lcd                       20 non-null     float64
 2   pld                       20 non-null     float64
 3   void_fraction             20 non-null     float64
 4   surface_area_m2g          20 non-null     float64
 5   mofid                     15 non-null     str    
 6   CO2_uptake_0.01bar_molkg  20 non-null     float64
 7   CO2_uptake_0.05bar_molkg  20 non-null     float64
 8   CO2_uptake_0.1bar_molkg   20 non-null     float64
 9   CO2_uptake_0.5bar_molkg   20 non

In [9]:
# process every JSON file this time, not just the first 20 samples
rows = [parse_one_mof(fp) for fp in json_files]

# convert the full list of parsed rows into one pandas DataFrame
df_full = pd.DataFrame(rows)

# print the shape (rows, columns) as a quick sanity check
print(df_full.shape)

# print column names, non-null counts, and data types for the full dataset
df_full.info()

(32768, 11)
<class 'pandas.DataFrame'>
RangeIndex: 32768 entries, 0 to 32767
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   filename                  32768 non-null  str    
 1   lcd                       32768 non-null  float64
 2   pld                       32768 non-null  float64
 3   void_fraction             32768 non-null  float64
 4   surface_area_m2g          32768 non-null  float64
 5   mofid                     29063 non-null  str    
 6   CO2_uptake_0.01bar_molkg  32768 non-null  float64
 7   CO2_uptake_0.05bar_molkg  32768 non-null  float64
 8   CO2_uptake_0.1bar_molkg   32768 non-null  float64
 9   CO2_uptake_0.5bar_molkg   32768 non-null  float64
 10  CO2_uptake_2.5bar_molkg   32768 non-null  float64
dtypes: float64(9), str(2)
memory usage: 2.8 MB


In [10]:
# build the output path for the full structural + CO2 dataset
output_path = Path("../data/raw/hmof_full_structural_co2.csv")

# write the full DataFrame to CSV, without including the pandas row index as a column
df_full.to_csv(output_path, index=False)

# confirm how many rows were saved and where
print(f"Saved {len(df_full)} rows to {output_path}")

Saved 32768 rows to ../data/raw/hmof_full_structural_co2.csv


In [11]:
# print the shape (rows, columns) of the full parsed dataset
print(df_full.shape)

# print column names, non-null counts, and data types for the full dataset
df_full.info()

(32768, 11)
<class 'pandas.DataFrame'>
RangeIndex: 32768 entries, 0 to 32767
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   filename                  32768 non-null  str    
 1   lcd                       32768 non-null  float64
 2   pld                       32768 non-null  float64
 3   void_fraction             32768 non-null  float64
 4   surface_area_m2g          32768 non-null  float64
 5   mofid                     29063 non-null  str    
 6   CO2_uptake_0.01bar_molkg  32768 non-null  float64
 7   CO2_uptake_0.05bar_molkg  32768 non-null  float64
 8   CO2_uptake_0.1bar_molkg   32768 non-null  float64
 9   CO2_uptake_0.5bar_molkg   32768 non-null  float64
 10  CO2_uptake_2.5bar_molkg   32768 non-null  float64
dtypes: float64(9), str(2)
memory usage: 2.8 MB
